# Exploratory Data Analysis (EDA) - Home Credit Default Risk

## Overview
This notebook provides a comprehensive exploratory analysis of the Home Credit Default Risk dataset. We examine data structure, distributions, missing values, and patterns to understand the dataset before model training.

## Dataset Information
- **Training Samples**: 307,511 loan applications
- **Test Samples**: 48,744 loan applications
- **Target Variable**: Whether applicant defaulted (0 = No, 1 = Yes)
- **Class Imbalance**: ~11.4% defaults

## EDA Objectives
1. Understand data structure and types
2. Identify missing values and patterns
3. Analyze feature distributions
4. Detect outliers and anomalies
5. Explore relationships between features and target


## 1. Loading and Inspecting Data

### Step 1.1: Import Libraries and Read Data
We use Polars for fast DataFrame operations and Plotly for interactive visualizations.


In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Load training data
train_df = pl.read_csv('../dataset/application_train.csv')
print('✓ Data loaded successfully!')
print(f'Dataset shape: {train_df.shape}')

### Step 1.2: Display Column Names and Data Types


In [ ]:
# Display column information
print(f'Total columns: {len(train_df.columns)}')
print('\nColumn names and types (first 20):')
for col in train_df.columns[:20]:
    print(f'  {col}: {train_df[col].dtype}')

### Step 1.3: Preview Data
Examine first few rows and basic statistics.


In [ ]:
# Display first few rows
print('First 5 rows:')
print(train_df.head())

In [ ]:
# Check data shape
print(f'Shape: {train_df.shape}')
print(f'Rows: {train_df.shape[0]:,}')
print(f'Columns: {train_df.shape[1]}')

In [ ]:
# Quick overview
print(train_df.glimpse())

In [ ]:
# Statistical summary
print(train_df.describe())

## 2. Data Quality Assessment

### Step 2.1: Check Data Types Distribution


In [ ]:
# Count data types
dtype_count = Counter(str(dtype) for dtype in train_df.dtypes)
print('Data type distribution:')
for dtype, count in sorted(dtype_count.items()):
    print(f'  {dtype}: {count} columns')

### Step 2.2: Check for Duplicates


In [ ]:
# Check duplicates
duplicates = train_df.is_duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')
print(f'Percentage: {(duplicates/len(train_df))*100:.2f}%')

### Step 2.3: Analyze Missing Values
Identify which features have missing data and their extent.


In [ ]:
# Check missing values count
missing_counts = train_df.null_count()
missing_cols = (missing_counts > 0).sum()
print(f'Columns with missing values: {missing_cols} out of {len(missing_counts)}')

In [ ]:
# Analyze missing value patterns
missing_analysis = []
for i, col in enumerate(train_df.columns):
    missing_count = train_df[col].null_count()
    if missing_count > 0:
        missing_pct = (missing_count / len(train_df)) * 100
        missing_analysis.append((col, missing_count, missing_pct))

# Sort by percentage
missing_analysis.sort(key=lambda x: x[2], reverse=True)
print('\nTop columns with missing values:')
for col, count, pct in missing_analysis[:10]:
    print(f'  {col}: {count:,} ({pct:.2f}%)')

In [ ]:
# Drop columns with >50% missing
cols_to_drop = [col for col, count, pct in missing_analysis if pct > 50]
print(f'Columns to drop (>50% missing): {len(cols_to_drop)}')
print(f'Before: {train_df.shape}')
train_df = train_df.drop(cols_to_drop)
print(f'After: {train_df.shape}')

## 3. Target Variable Analysis

### Step 3.1: Class Distribution
Understanding the balance of default vs non-default cases.


In [ ]:
# Target distribution analysis
target_count = train_df['TARGET'].value_counts().sort('counts', descending=True)
print('Target variable distribution:')
total = target_count['counts'].sum()

for row in target_count.iter_rows(named=True):
    target_val = int(row['TARGET'])
    count = row['counts']
    pct = (count / total) * 100
    label = 'Default' if target_val == 1 else 'Non-Default'
    print(f'  {label}: {count:,} ({pct:.2f}%)')

In [ ]:
# Visualize target distribution
target_df = train_df['TARGET'].value_counts().sort('TARGET').to_pandas()
fig = px.bar(
    target_df,
    x='TARGET',
    y='counts',
    labels={'TARGET': 'Default Status (0=No, 1=Yes)', 'counts': 'Count'},
    title='Target Distribution - Class Imbalance Analysis',
    text='counts',
    color='TARGET'
)
fig.update_xaxes(tickmode='linear', tick0=0)
fig.show()

## 4. Feature Analysis

### Step 4.1: Categorize Numerical and Categorical Features


In [ ]:
# Separate features by type
numerical_columns = []
categorical_columns = []

for col in train_df.columns:
    if col != 'TARGET':
        dtype_str = str(train_df[col].dtype)
        if 'Int' in dtype_str or 'Float' in dtype_str:
            numerical_columns.append(col)
        else:
            categorical_columns.append(col)

print(f'Numerical features: {len(numerical_columns)}')
print(f'Categorical features: {len(categorical_columns)}')
print(f'\nSample numerical: {numerical_columns[:5]}')
print(f'Sample categorical: {categorical_columns[:5]}')

### Step 4.2: Numerical Features Distribution


In [ ]:
# Analyze numerical features
for col in numerical_columns[2:8]:
    print(f'\n{col}:')
    print(f'  Mean: {train_df[col].mean():.2f}')
    print(f'  Median: {train_df[col].median():.2f}')
    print(f'  Std: {train_df[col].std():.2f}')
    print(f'  Min: {train_df[col].min():.2f}')
    print(f'  Max: {train_df[col].max():.2f}')

In [ ]:
# Histogram plots for numerical features
for col in numerical_columns[2:8]:
    fig = px.histogram(
        train_df.to_pandas(),
        x=col,
        nbins=50,
        title=f'Distribution of {col}',
    )
    fig.show()

### Step 4.3: Categorical Features Distribution


In [ ]:
# Bar plots for categorical features
for col in categorical_columns[:5]:
    count_df = train_df[col].value_counts().sort('counts', descending=True).head(10)
    fig = px.bar(
        count_df.to_pandas(),
        x=col,
        y='counts',
        title=f'Top Categories - {col}',
        text='counts'
    )
    fig.show()

### Step 4.4: Outlier Detection via Box Plots


In [ ]:
# Box plots to identify outliers
for col in numerical_columns[2:8]:
    fig = px.box(
        train_df.to_pandas(),
        y=col,
        title=f'Box Plot - {col}',
    )
    fig.show()

## 5. Summary of Key Findings

### Insights:
1. **Class Imbalance**: 11.4% defaults vs 88.6% non-defaults
   - Requires careful handling (class weights, sampling strategies)

2. **Missing Values**: 
   - Identified columns with >50% missing values
   - Dropped for better model performance

3. **Feature Types**:
   - Mix of numerical and categorical features
   - Different preprocessing needed for each type

4. **Outliers**: 
   - Detected through box plots
   - May need handling via winsorization or removal

5. **Data Quality**: 
   - No significant duplicate records
   - Ready for feature engineering

### Next Steps in Pipeline:
- Feature engineering from related tables (Bureau, Previous Applications)
- Data scaling and normalization
- Handling class imbalance
- Model training and evaluation
- Hyperparameter optimization
